# 05 Embedder Tiny Training

Goal: run the first tiny triplet-loss training loop in-notebook. This is intentionally small. For meaningful learning, point `TRIPLETS_PATH` at real neutral drafts, not `triplets_dry_run.jsonl`.

If dependencies are missing, run this once. It works even if the remote kernel is not started from the repo root:

```python
import sys, subprocess
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
req = next((p / "requirements.txt" for p in candidates if (p / "requirements.txt").exists()), None)
cmd = [sys.executable, "-m", "pip", "install"]
cmd += ["-r", str(req)] if req else ["torch", "transformers", "openai", "python-dotenv"]
subprocess.check_call(cmd)
```

In [ ]:
import os
from pathlib import Path
import sys

def find_project_root():
    env_root = os.getenv("VOICE_CLONE_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "voice_clone").exists():
            return candidate
    raise RuntimeError("Could not find project root. Set VOICE_CLONE_ROOT to the repo path in this kernel.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from voice_clone.data.dataset import TripletTextDataset
from voice_clone.models import StyleEmbedder, count_parameters

TRIPLETS_PATH = ROOT / "data" / "processed" / "triplets_dry_run.jsonl"
MODEL_NAME = "allenai/longformer-base-4096"
BATCH_SIZE = 1
MAX_LENGTH = 512
STEPS = 5
MARGIN = 0.25

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("ROOT:", ROOT)
print("TRIPLETS_PATH:", TRIPLETS_PATH)
device

In [ ]:
dataset = TripletTextDataset.from_jsonl(TRIPLETS_PATH, seed=7)
sample = dataset[0]
if sample["negative"].startswith("[DRY RUN"):
    print("WARNING: dry-run negatives are placeholders. This checks mechanics only, not learning quality.")

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
len(dataset), sample["author_id"], sample["anchor_doc_id"], sample["positive_doc_id"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = StyleEmbedder.from_pretrained(MODEL_NAME).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

count_parameters(model)

In [ ]:
def encode_texts(texts):
    encoded = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    return {k: v.to(device) for k, v in encoded.items()}


def triplet_cosine_loss(anchor, positive, negative, margin=MARGIN):
    positive_distance = 1 - torch.sum(anchor * positive, dim=-1)
    negative_distance = 1 - torch.sum(anchor * negative, dim=-1)
    return F.relu(positive_distance - negative_distance + margin).mean()

In [ ]:
model.train()
iterator = iter(loader)

for step in range(1, STEPS + 1):
    batch = next(iterator)
    texts = list(batch["anchor"]) + list(batch["positive"]) + list(batch["negative"])
    encoded = encode_texts(texts)

    embeddings = model(**encoded)
    anchor, positive, negative = embeddings.chunk(3, dim=0)
    loss = triplet_cosine_loss(anchor, positive, negative)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        pos_sim = torch.sum(anchor * positive, dim=-1).mean().item()
        neg_sim = torch.sum(anchor * negative, dim=-1).mean().item()
    print(f"step={step} loss={loss.item():.4f} pos_sim={pos_sim:.4f} neg_sim={neg_sim:.4f}")